# GPU Performance Analysis

This tutorial will explore how to evaluate and optimize the performance of GPU-accelerated codes.
The goal is to:
- Understand the expected optimal performance of your code.
- Identify how far the actual performance deviates from the optimal.
- Explore reasons for performance gaps and practical methods to bridge them.

The material is based on a distilled version of the NHR@FAU workshop GPU Performance Engineering and designed for a two-hour event.
Feel free to check out the material on [github](https://github.com/SebastianKuckuk/gpu-performance-engineering).
Dates for upcoming public iterations for the European region are listed on the [training overview page](https://hpc.fau.de/teaching/tutorials-and-courses/) of NHR@FAU.

# Introduction

Throughout this tutorial we follow the (slightly exaggerated and mildly cartoonish) journey of a fictitious PhD candidate in computer science.
Any resemblance to actual experiences is, of course, purely coincidental.

Our protagonist has just inherited a piece of legacy code: something their supervisor wrote a while back that now, at long last, needs to be published.
The catch is that it is CPU-only, and CPU-only is simply not *en vogue* anymore.
So the assignment lands on their desk: *port it to the GPU*.
After all, GPUs are all about performance and a shiny speed-up always sells.

Along the way we meet the recurring cast of their research group: the **supervisor** with strong opinions and vague technical specifics, the **wise colleague** who has seen it all before, and, doing the actual work, **our PhD candidate**.

# Our Test Application - 2D Stencil Updates

## CPU Baseline

Like any sensible engineer, our PhD candidate resists the temptation to change anything just yet.
Step one is to read the inherited code and run it exactly as handed over, establishing a baseline against which all future glory can be measured.

They soon realize that the code is basically a simple yet widely used benchmark: a 2D stencil application.
It can be regarded as a proxy application for (matrix-free) matrix-vector multiplications, which are ubiquitous in HPC applications.

A serial baseline *CPU-only* implementation can be found in [stencil-2d-base.cpp](../src/stencil-2d/stencil-2d-base.cpp).
Reviewing the implementation reveals these key points:
* The main workload is encapsulated in the `stencil2d` function.
* The application can be parameterized with command line arguments (c.f. `parseCLA_2d` in [stencil-2d-util.h](../src/stencil-2d/stencil-2d-util.h)):
  - **Data type**: `float` or `double`
  - **nx, ny**: Grid dimensions, defining total workload (`nx * ny`)
  - **nWarmUp**: Number of non-timed warm-up iterations
  - **nIt**: Number of timed iterations
* Basic diagnostic output of performance data is available via the `printStats` function in [util.h](../src/util.h).
  - Key **performance metrics** such as the sustained bandwidth are *estimated*

After reviewing the code, our PhD candidate uses the following commands to compile and execute the application:

In [ ]:
!g++ -O3 -march=native -std=c++17 ../src/stencil-2d/stencil-2d-base.cpp -o ../build/stencil-2d-base

In [ ]:
!../build/stencil-2d-base double 8192 8192 2 256

Their supervisor drops by with a hint: a serial baseline is *wonderful* for advertising maximal speed-ups after the port, but reviewers have caught on to that particular magic trick and will invariably also ask for a *parallel* CPU baseline.
Fair enough.
Our PhD candidate dusts off the memories of that one OpenMP course they took a while back and applies it to parallelize the code on the CPU first.

The updated version is available in [stencil-2d-omp-host.cpp](../src/stencil-2d/stencil-2d-omp-host.cpp), and can be compiled and executed using:

In [ ]:
!g++ -O3 -march=native -std=c++17 -fopenmp ../src/stencil-2d/stencil-2d-omp-host.cpp -o ../build/stencil-2d-omp-host

In [ ]:
!../build/stencil-2d-omp-host double 8192 8192 2 256

Depending on the parameters and the hardware, the gains may be substantial, modest, entirely absent, or, for a particularly unlucky combination of factors, even negative.

## First Attempt at GPU Acceleration

Emboldened by the CPU version, our PhD candidate recalls another nugget from that same OpenMP course: OpenMP can also target GPUs, via *target offloading*.
Same framework, new hardware: how hard could it be?
They decide to find out just how far a few extra pragmas will take them.

To offload computations to the GPU, we extend the code with OpenMP target offloading.
In a first attempt, we limit code changes to the `stencil2d` function.
The updated version is in [stencil-2d-omp-target-v0.cpp](../src/stencil-2d/stencil-2d-omp-target-v0.cpp).

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v0.cpp -o ../build/stencil-2d-omp-target-v0

In [ ]:
!../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

The results are... not what they expected.
Our PhD candidate hoped for a triumphant speed-up; instead the GPU version is *slower* than the CPU baseline.
This is decidedly not publishable.
When they bring the bad news to their supervisor, the response is not sympathy but a question they had not thought to ask themselves: *"And what performance were you actually expecting?"*

Surprisingly, or entirely predictably if you have used OpenMP target offloading before, this first GPU attempt is slower than the CPU baseline.
For the sake of the story, let's play along and pretend we have no idea why.

There is a silver lining: we can already *evaluate* performance, which is useful for comparing different variants of the same application.
What we still cannot answer, though, are exactly the questions the supervisor's raised eyebrow implied:
* Could performance be better for this particular hardware and software combination?
* If so, how do we pinpoint which optimizations to apply, and where, to raise our performance levels?

# Performance Models

"What performance *should* I expect?" turns out to be the right question, and our PhD candidate has no idea how to answer it.
A dim memory surfaces of the supervisor once muttering something about *performance models*.
Lacking a more precise reference, they do what any of us would: a quick round of online research.

Performance models aim at relating algorithms, their implementation, hardware characteristics and the expected performance (execution time).
One of the most straightforward ones is the *bottleneck model*.

## Bottleneck Model

The bottleneck model analyzes potential performance limiters individually, such as:
* **DRAM Bandwidth**, i.e. the speed at which data can be transferred from/to global memory.
* **Compute Throughput**, i.e. the rate at which computations can be executed.
* **Cache Bandwidth**, i.e. the speed of accessing data from different parts of the memory hierarchy, e.g. L2 cache, L1 cache, or shared memory.

After each bottleneck was modeled separately, their individual contributions are compared to find the dominant one.
This *main bottleneck* can then be used to estimate the expected performance and thereby also the expected execution time under ideal conditions - the lightspeed estimate. 

To put it into different words: application performance is generally limited by the time required to do the meaningful computations, or by getting data to and from where the compute is happening.
Whichever takes longer determines overall execution time.

# Hardware Characteristics

Suddenly those strange numbers the inherited code prints at the end make sense: they are the raw material a performance model feeds on.
To compare them against what the hardware should deliver, our PhD candidate goes looking for the specifications of the GPUs available to them.

The bottleneck model, like other performance models, applies to both CPUs and GPUs.
Since this course focuses on GPU performance, we will examine the theoretical limits of key GPU bottlenecks.
Relevant information can be found in official data sheets and white papers ([V100](https://images.nvidia.com/content/volta-architecture/pdf/volta-architecture-whitepaper.pdf), [A100](https://images.nvidia.com/aem-dam/en-zz/Solutions/data-center/nvidia-ampere-architecture-whitepaper.pdf), [H100](https://resources.nvidia.com/en-us-hopper-architecture/nvidia-h100-tensor-c)), system documentation, third-party resources (such as [TechPowerUp](https://www.techpowerup.com/gpu-specs/)), and obtained by running micro-benchmarks (see [below](#micro-benchmarks)).

|                             |  V100 SXM2 32 GB   | A100 SXM 40 GB \| 80 GB | H100 SXM 94 GB HBM2e  | H100 SXM5 80 GB HBM3  |
| --------------------------- | :----------------: | :---------------------: | :-------------------: | :-------------------: |
| Availability                |     Bridges-2      |      NHR@FAU Alex       |     NHR@FAU Helma     |       Bridges-2       |
| CUDA                        |        7.0         |           8.0           |          9.0          |          9.0          |
| #Cores                      | 5120<br/>(80 * 64) |   6912<br/>(108 * 64)   | 16896<br/>(132 * 128) | 16896<br/>(132 * 128) |
| FP32 Performance \[TFLOPS\] |         16         |           19            |          67           |          67           |
| FP64 Performance \[TFLOPS\] |         8          |          9.75           |          33           |          33           |
| FP64:FP32 Ratio             |        1:2         |           1:2           |          1:2          |          1:2          |
| Memory \[GB\]               |         32         |        40 \| 80         |          94           |          80           |
| Bandwidth \[GB/s\]          |        898         |      1555 \| 2039       |         2400          |         3360          |
| L2 Cache \[MB\]             |         6          |           40            |          50           |          50           |
| TDP \[W\]                   |        300         |           400           |          700          |          700          |

Comparing our first attempt against these numbers, it is painfully evident that we hit *none* of the major performance limits of the GPU: we are not even in the same order of magnitude.
The upside is that there is clearly a lot of room to improve.

# Application Level Profiling

Armed with a performance model, our PhD candidate now knows they are (multiple) orders of magnitude away from the hardware limits, but not *why*.
Puzzled, they wander over to a more experienced colleague for advice.
The verdict is immediate and, to anyone who has done this before, entirely unsurprising: *stop guessing and start profiling.*

We follow a *top-down approach* to performance analysis, starting with a *whole application* performance overview and then narrowing down to specific *hot spots*.
An initial overview can be obtained using Nsight Systems, using either solely the command line interface, or by complementing the analysis with the provided GUI.

## Nsight Systems CLI

First, we compile and execute our benchmark application to make sure that results are as expected.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v0.cpp -o ../build/stencil-2d-omp-target-v0
!../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

Next, we profile our binary with `nsys profile`.
The most important command line arguments are:
* `--stats=true`: prints a summary of performance statistics on the command line
* `-o ...`: sets the target output profile file
* `--force-overwrite=true`: replaces the profile file if it already exists (instead of aborting)

We also configure what we are interested in with:
* `--trace=cuda`: logs CUDA API calls, host-device memory transfers, kernels, ...
* `--gpu-metrics-devices cuda-visible`: collects GPU metrics for all visible devices

A full list of available arguments for the profile mode is available as part of the official [documentation](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#cli-profile-command-switch-options).

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-omp-target-v0 --force-overwrite=true ../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

The output of the command line is organized in multiple categories.
A possible output for an NVIDIA A100 (40 GB) is copied in below (truncated for conciseness and readability):

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  ------------  --------------------
     59.4   28,658,756,724        516  55,540,226.2  58,390,100.5  41,275,285  72,668,977  11,131,385.8  cuMemcpyDtoHAsync_v2
     40.6   19,610,023,579        516  38,003,921.7  37,950,273.5  37,646,652  44,313,738     353,887.2  cuMemcpyHtoDAsync_v2
      0.0        7,414,219        258      28,737.3      26,528.5      22,095      51,803       5,641.2  cuLaunchKernel      
      0.0        1,815,349          5     363,069.8     213,747.0      75,321     780,811     304,899.1  cuMemFree_v2        
      0.0        1,727,112          4     431,778.0     464,638.5      12,575     785,260     387,811.9  cuMemAlloc_v2       
      0.0        1,384,138        258       5,364.9       5,301.0       3,998       8,817         582.8  cuStreamSynchronize 
      0.0        1,143,576          1   1,143,576.0   1,143,576.0   1,143,576   1,143,576           0.0  cuMemAllocHost_v2   
      0.0          652,885          1     652,885.0     652,885.0     652,885     652,885           0.0  cuMemFreeHost       
      0.0          575,119          1     575,119.0     575,119.0     575,119     575,119           0.0  cuModuleLoadDataEx  
      0.0          168,747          1     168,747.0     168,747.0     168,747     168,747           0.0  cuMemAllocManaged   
      0.0            6,152          5       1,230.4         872.0         230       2,946       1,143.9  cuCtxSetCurrent     
      0.0            1,402          1       1,402.0       1,402.0       1,402       1,402           0.0  cuInit              
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                     Name                    
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  -------------------------------------------
    100.0    1,623,550,587        258  6,292,831.7  6,277,925.0  6,262,783  8,105,030    159,852.3  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)            Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  ------------  ----------------------------
     57.9   26,968,995,312    516  52,265,494.8  55,135,060.5  41,185,538  67,014,017  10,528,079.7  [CUDA memcpy Device-to-Host]
     42.1   19,594,388,397    516  37,973,620.9  37,921,186.0  37,608,700  44,272,201     353,441.1  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)   Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 -----------  -----  --------  --------  --------  --------  -----------  ----------------------------
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

The first wall of numbers is genuinely overwhelming: columns of nanoseconds and byte counts that mean nothing at first glance.
Fortunately the wise colleague is still within earshot, and together they sift the noise from the signal and pull out a handful of key insights.

Looking at the statistics they identify multiple effects:
* The number of kernel instances matches our expectation ($2 + 256$).
* The number of memory transfers seems to be related to the number of kernel instances.
* Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.
* Comparing aggregated memory transfer and kernel execution times reveals a stark difference.
  * \> Even if the kernel could be accelerated, overall performance will most likely not increase.
  * \> These numbers could be used to approximate a minimum number of iterations at which the memory transfers get amortized (assuming that the transfer times *don't scale with the number of iterations*).
* The size per transfer matches our expectation ($8192^2 \cdot 8 B \approx 537 MB$).

## Nsight Systems GUI

One thing nags at our PhD candidate: what *are* all those memory transfers doing there?
Numbers in a table only go so far, so they open the generated report in the graphical interface to see the timeline with their own eyes.

We take a closer look at the problematic memory transfers by opening the generated `stencil-2d-omp-target-v0.nsys-rep` report file located in the `profiles` folder.
You can download it directly from the folder, or **shift + right-click** on this [link](../profiles/stencil-2d-omp-target-v0.nsys-rep) and select "Download".
Afterwards, open it up in your local installation of Nsight Systems.

As you may have already suspected, the timeline shows a recurring pattern of
* two memory transfers (HtoD),
* a kernel call, and
* two memory transfers (DtoH)

Additionally, `cudaStreamSynchronize` (and thereby `cuStreamSynchronize`) is only called at the end of this pattern, which partly explains the deviation from the kernel execution time.

At this stage, our PhD candidate has two options:
* Update the bottleneck model to explicitly include PCIe transfers, or
* Optimize the application to eliminate unnecessary data transfers.

If we choose the first option, we need to consider the peak bandwidth for the PCIe connection, which depends on the GPU generation:

|                                    |  Volta  | Ampere  | Hopper  | 
|------------------------------------|:-------:|:-------:|:-------:|
| PCIe                               | 3.0 x16 | 4.0 x16 | 5.0 x16 |
| Bandwidth (per direction) \[GB/s\] |  15.8   |  31.5   |  63.0   |

Our PhD candidate decides that explainable bad performance is better than inexplicable bad performance, but definitely less attractive than fixing the performance bug altogether.
They thus focus on reducing redundant transfers.

## Stencil Code Optimization - Data Transfers

Back to the course notes.
Buried in the OpenMP target offloading material, right where all important warnings live, i.e. in a box our PhD candidate skimmed the first time around, is the hint they needed all along: *minimize host-device data transfers; ideally move the data over once at the start and back once at the end.*

Having pinpointed our performance bug, we can now optimize data transfers in our application.
One straightforward way is adding unstructured data primitives in our code, basically spanning a region at whose begin and end data is copied *only once*.
The updated version is available at [stencil-2d-omp-target-v1.cpp](../src/stencil-2d/stencil-2d-omp-target-v1.cpp), and can be compiled, executed and profiled using the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1.cpp -o ../build/stencil-2d-omp-target-v1

In [ ]:
!../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-omp-target-v1 --force-overwrite=true ../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)          Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  --------------------
     90.5    1,647,854,199        260   6,337,900.8   6,271,558.0       6,503   8,091,058    708,087.2  cuStreamSynchronize 
      4.8       87,611,154          2  43,805,577.0  43,805,577.0  42,702,039  44,909,115  1,560,638.4  cuMemcpyDtoHAsync_v2
      4.2       76,852,527          2  38,426,263.5  38,426,263.5  37,968,978  38,883,549    646,699.4  cuMemcpyHtoDAsync_v2
      0.1        2,112,576          4     528,144.0     469,013.0       9,489   1,165,061    528,701.8  cuMemAlloc_v2       
      0.1        1,911,153          5     382,230.6     210,812.0      83,547     845,331    322,507.2  cuMemFree_v2        
      0.1        1,186,333          1   1,186,333.0   1,186,333.0   1,186,333   1,186,333          0.0  cuMemAllocHost_v2   
      0.1        1,026,079        258       3,977.1       3,492.0       3,076      28,728      2,361.9  cuLaunchKernel      
      0.0          672,856          1     672,856.0     672,856.0     672,856     672,856          0.0  cuMemFreeHost       
      0.0          576,874          1     576,874.0     576,874.0     576,874     576,874          0.0  cuModuleLoadDataEx  
      0.0          161,844          1     161,844.0     161,844.0     161,844     161,844          0.0  cuMemAllocManaged   
      0.0            6,063          5       1,212.6         491.0         220       2,976      1,223.2  cuCtxSetCurrent     
      0.0            1,272          1       1,272.0       1,272.0       1,272       1,272          0.0  cuInit              
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                     Name                    
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  -------------------------------------------
    100.0    1,646,336,131        258  6,381,147.8  6,266,257.0  6,246,704  8,092,982    438,539.5  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     53.1       86,934,668      2  43,467,334.0  43,467,334.0  42,164,690  44,769,978  1,842,216.8  [CUDA memcpy Device-to-Host]
     46.9       76,750,542      2  38,375,271.0  38,375,271.0  37,928,486  38,822,056    631,849.4  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

Looking at the statistics we can now compare the effects previously discussed:
* The number of kernel instances matches our expectation ($2 + 256$).
* ~~The number of memory transfers seems to be related to the number of kernel instances.~~
* ~~Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.~~
* ~~Comparing aggregated memory transfer and kernel execution times reveals an order of magnitude in difference.~~
* The size per transfer matches our expectation ($8192 \cdot 8192 \cdot 8 \text{B} \approx 537 \text{MB}$).


Opening up the generated profile output in our GUI reveals what we already expected: single staging parts at the beginning and the end of our application with multiple kernel calls in between.
The output file is once again collected in the `../profiles` folder and accessible via [link](../profiles/stencil-2d-omp-target-v1.nsys-rep).

Finally, a speed-up over the CPU baseline, and a real one this time.
Under normal circumstances this is the moment our PhD candidate would open a fresh document titled *paper_final_v1* and start typing.
But the performance model is still whispering: we remain a comfortable factor away from the hardware limits.
And where there is a gap, there is a bigger, shinier speed-up waiting to be claimed.

# Kernel Level Profiling

Another trip to the wise colleague's desk.
Their advice: now that the profiler confirms that *kernel execution* is the main hotspot, and no longer anything else such as data transfers, it is time to zoom in and put these kernels under the microscope.
The tool for that job is Nsight Compute.

The next step would usually be to use the application-level analysis to pinpoint the most critical kernel(s), i.e. those that contribute most to the overall execution time, and to focus on them first.
In our case, this step is straightforward: the application contains only a single kernel.

We profile the kernel performance of our application using the CLI of Nsight Compute: `ncu`.
Key command line arguments are (all command line arguments are listed in the [documentation](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html#profile)):
* `-o ...`: sets the target output profile file (equivalent to `nsys`)
* `--force-overwrite`: replaces the profile file if it already exists (in contrast to `nsys` no `=true`)

By default *every kernel* is profiled.
Apart from adapting the number of iterations in our applications, we can further limit the scope of profiled kernels with
* `--launch-skip n` or `-s n`: skips the first `n` kernels encountered
* `--launch-count n` or `-c n`: limits profiling to the first `n` applicable kernels
* `--kernel name` or `-k name`: limits profiling to kernels with the name `name`
  * can also be used with regex, e.g. `-k regex:"stencil[1-3]d"`
  * Nsight Compute also supports kernel renaming

In [ ]:
!ncu --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-omp-target-v1 --force-overwrite ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

If no output file is specified, key profiling insights will be displayed directly in the command line output.

In [ ]:
!ncu --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (64, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.21
    SM Frequency                    Ghz         1.10
    Elapsed Cycles                cycle    8,843,547
    Memory Throughput                 %        35.17
    DRAM Throughput                   %         8.66
    Duration                         ms         8.07
    L1/TEX Cache Throughput           %        60.91
    L2 Cache Throughput               %        23.08
    SM Active Cycles              cycle 5,112,631.43
    Compute (SM) Throughput           %         1.10
    ----------------------- ----------- ------------

    OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.05 full     
          waves across all SMs. Look at Launch Statistics for more details.                                             
```

```
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   128
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                     64
    Registers Per Thread             register/thread              36
    Shared Memory Configuration Size           Kbyte           32.77
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM             108
    Stack Size                                                 1,024
    Threads                                   thread           8,192
    # TPCs                                                        54
    Enabled TPC IDs                                              all
    Uses Green Context                                             0
    Waves Per SM                                                0.05
    -------------------------------- --------------- ---------------

    OPT   Est. Speedup: 40.74%                                                                                          
          The grid for this launch is configured to execute only 64 blocks, which is less than the 108 multiprocessors  
          used. This can underutilize some multiprocessors. If you do not intend to execute this kernel concurrently    
          with other workloads, consider reducing the block size to have at least one block per multiprocessor or       
          increase the size of the grid to fully utilize the available hardware resources. See the Hardware Model       
          (https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-hw-model) description for more      
          details on launch configurations.                                                                             
```

```
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           32
    Block Limit Registers                 block           12
    Block Limit Shared Mem                block           32
    Block Limit Warps                     block           16
    Theoretical Active Warps per SM        warp           48
    Theoretical Occupancy                     %           75
    Achieved Occupancy                        %         6.25
    Achieved Active Warps Per SM           warp         4.00
    ------------------------------- ----------- ------------

    OPT   Est. Local Speedup: 91.67%                                                                                    
          The difference between calculated theoretical (75.0%) and measured achieved occupancy (6.2%) can be the       
          result of warp scheduling overheads or workload imbalances during the kernel execution. Load imbalances can   
          occur between warps within a block as well as across blocks of the same kernel. See the CUDA Best Practices   
          Guide (https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#occupancy) for more details on     
          optimizing occupancy.                                                                                         
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Local Speedup: 25%                                                                                       
          The 12.00 theoretical warps per scheduler this kernel can issue according to its occupancy are below the      
          hardware maximum of 16. This kernel's theoretical occupancy (75.0%) is limited by the number of required      
          registers.                                                                                                    
```

```
    Section: GPU and Memory Workload Distribution
    -------------------------- ----------- -------------
    Metric Name                Metric Unit  Metric Value
    -------------------------- ----------- -------------
    Average DRAM Active Cycles       cycle    849,020.90
    Total DRAM Elapsed Cycles        cycle   392,090,112
    Average L1 Active Cycles         cycle  5,112,631.43
    Total L1 Elapsed Cycles          cycle   956,315,032
    Average L2 Active Cycles         cycle  8,457,989.21
    Total L2 Elapsed Cycles          cycle   677,669,040
    Average SM Active Cycles         cycle  5,112,631.43
    Total SM Elapsed Cycles          cycle   956,315,032
    Average SMSP Active Cycles       cycle  5,112,072.00
    Total SMSP Elapsed Cycles        cycle 3,825,260,128
    -------------------------- ----------- -------------

    OPT   Est. Speedup: 24.31%                                                                                          
          One or more SMs have a much lower number of active cycles than the average number of active cycles. Maximum   
          instance value is 42.11% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 24.38%                                                                                          
          One or more SMSPs have a much lower number of active cycles than the average number of active cycles. Maximum 
          instance value is 42.24% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 24.31%                                                                                          
          One or more L1 Slices have a much lower number of active cycles than the average number of active cycles.     
          Maximum instance value is 42.11% above the average, while the minimum instance value is 100.00% below the     
          average.                                                                                                      
```

### Interpretation

The first section, *GPU Speed Of Light Throughput*, provides a summary of key metrics and potential bottlenecks.
The most relevant for our analysis are:
* `Duration` of a single kernel execution (in ms)
* `DRAM Throughput` as a percentage of the theoretical maximum
* `Compute (SM) Throughput` as a percentage of the theoretical maximum

By default, Nsight Compute also offers suggestions for improving performance.
For example, we might see output such as:
```
OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.05 full     
      waves across all SMs. Look at Launch Statistics for more details.                                             
```

Overwhelmed (again), our PhD candidate decides to simply start at the top of this list of helpful suggestions.
The very first one stops them cold: something about the grid being too small to fill the device with enough *waves*.
What on earth is a wave?
To make sense of it, and in general of what is happening on the GPU when work is offloaded, they set out to understand how a modern GPU is actually put together.

# GPU Architecture 101

## Example: NVIDIA H100

<img align="right" src="img/h100-chip.png" alt="H100 Chip" width="384px"/>

A fundamental design principle of modern GPUs is their hierarchical structure, which enables high levels of parallelism and scalability.
To better understand GPU performance, we will examine the NVIDIA H100 architecture as a representative example of state-of-the-art GPUs.

Further details can be found in the official [NVIDIA H100 White Paper](https://resources.nvidia.com/en-us-tensor-core).
The following figures and information are also derived from this resource.

![H100 Chip Layout with Annotations](img/h100-layout.png)

The diagram above depicts the 'full configuration' of a single NVIDIA H100 chip.
However, not all functional units shown are accessible in production GPUs.
For example, the SXM5 version of the H100 features **132 Streaming Multiprocessors (SMs)** enabled for computation.

<img align="right" src="img/h100-sm-layout.png" alt="H100 SM Layout" width="384px" style="background-color:white"/>

Each SM is subdivided into 4 sub partitions (SMSP), as visualized in the figure on the right, each with
* 16 INT32 units, for a total of $132 \cdot 4 \cdot 16 = 8448$,
* 32 FP32 units, for a total of $132 \cdot 4 \cdot 32 = 16896$,
* 16 FP64 units, for a total of $132 \cdot 4 \cdot 16 = 8448$, and
* 1 tensor core, for a total of $132 \cdot 4 = 528$.

Each unit is capable of executing one fused-multiply-add (FMA) operation per cycle, with the exception of the tensor cores which can each perform 512 FP16/FP32-mixed-precision FMAs at once.

## Memory Hierarchy

<img align="right" src="img/h100-memory-abstract.png" alt="memory abstraction" width="512px"/>

The GPU memory system is organized into a hierarchy designed to balance capacity, speed, and access latency.
Its levels, at the example of the H100, are (latency and bandwidth values provided by [GPU Benches](https://github.com/RRZE-HPC/gpu-benches))
- **Registers**
    - Scope: Private to each thread.
    - Latency: Typically 1 cycle.
    - Capacity: Each SM has 256 KB available for registers.
- **L1 Cache** and **Shared Memory**
    - Scope: Shared among all threads scheduled on a given SM. Threads of a given block can access the same shared memory.
    - Latency: ~32 cycles.
    - Capacity: Configurable up to 100 KB per SM for shared memory, with the remaining space used for L1 cache (up to 256 KB per SM).
- **L2 Cache**
    - Scope: Shared among all threads (GPU-wide)
    - Latency: ~280 cycles.
    - Capacity: 50 MB (two partitions with 25 MB each)
- **Global Memory (DRAM)**
    - Scope: Global to all threads.
    - Latency: ~690 cycles.
    - Capacity: between 80 and 94 GB
    - Bandwidth: between 2 and 3.9 TB/s.

## Waves and Occupancy

While this detour into GPU architecture was interesting, we still have an open question: what is a *wave*?

To answer this, let's clarify how threads are organized on GPUs:
- **OpenMP** organizes *threads* in *teams*.
- **CUDA** organizes *threads* in *blocks*.

Although OpenMP teams and CUDA thread blocks are not strictly equivalent, compilers typically map teams to blocks when targeting NVIDIA GPUs.
These blocks are then assigned to Streaming Multiprocessors (SMs):
* Each block is scheduled to one SM.
* Each SM can handle multiple blocks concurrently (depending on available resources).

Within a block, threads are further grouped into *warps*: groups of 32 threads.
Warps are the basic unit of scheduling on the SM sub-partitions (SMSPs).

A *wave* refers to the maximum number of blocks that can be resident on the GPU at the same time.
It is calculated as the number of SMs times the maximum number of blocks per SM.

Having at least one full wave ensures that all resources are fully utilized.
Multiple (full) waves work as well, but partial waves may lead to sub-optimal performance.

This closely relates to the concept of *occupancy* which is defined as the ratio of active warps to the maximum number of warps that can be scheduled on an SM.

# Micro Benchmarks

The architecture detour makes one thing clear: a GPU is hungry for parallelism, and lots of it.
But *how much* is enough?

Our PhD candidate faces two fundamental ways of approaching this question:

1. **Conceptual Analysis:** \
To have a GPU fully utilize its computational resources, it requires at least as many active threads as there are execution units for the current data type - typically on the order of ten thousand.
However, most applications require additional data transfers between main memory (or at least L2 cache) and the compute units, which introduces high latency.
To hide this latency and keep the GPU busy, resources need to be oversubscribed: when some warps are stalled waiting for data, others can be scheduled to execute.
In practice this usually relates to an order of magnitude more threads required (on the order of a hundred thousand).

2. **Benchmarking:** \
(Micro) benchmarks can help study performance for varying number of threads.
They can additionally be used to examine the effects of different data types, memory access patterns, computational intensities, the ratio of compute to data transfer, and more.

Rather than argue about it in the abstract, our PhD candidate decides to measure with a couple of micro-benchmarks they found on GitHub in the [Accelerated Programming EXamples (APEX)](https://github.com/SebastianKuckuk/apex) collection.
They choose two flavors: a [fused-multiply-add (FMA)](https://github.com/SebastianKuckuk/apex/tree/main/src/benchmark/fma) benchmark to assess computational throughput, and a [copy-stream](https://github.com/SebastianKuckuk/apex/tree/main/src/benchmark/stream) benchmark to evaluate main memory bandwidth.
Automated benchmarking of both is luckily available as part of the [APEX-Generator](https://github.com/SebastianKuckuk/apex-generator) project.

## FMA Benchmark

![FMA Benchmark Results for A100 80GB](img/A100-SXM4-80GB---fma.png)

Here we can see the results for the FMA benchmark on a single A100 80GB and the following effects:
* The maximum performance observed is $9.7$ TFLOP/s.
    * This is pretty much the theoretical limit given by the data sheet for double precision.
* A distinct saw-tooth pattern is visible.
    * The first peak is at $26\,615$ threads, the first drop is at $28\,526$ threads. 
    * This is just around $256 \cdot 108 = 27\,648$ threads (using a block size of 256 threads, as chosen by the CUDA FMA benchmark), i.e. when each SM gets exactly one thread block.
    * At the first drop, performance nearly halves because the amount of work increases only slightly, but the execution time doubles due to one SM having doubled workload. 
* Compared to the CUDA version, the compiler seems to adapt block sizes when using OpenMP target offloading. 

## Stream Benchmark

![Stream Benchmark Results for A100 80GB](img/A100-SXM4-80GB---stream.png)

Here we can see the results for the stream benchmark on a single A100 80GB and the following effects:
* The asymptotic bandwidth is around $1\,750$ GB/s, which is just around $85\%$ of the theoretical limit of $2\,040$ GB/s.
* The maximum bandwidth is around $2\,580$ GB/s for $1\,290\,948$ threads.
    * Each thread moves 16 bytes in the double precision case.
    * This corresponds to a total volume of just over $20$ MB, one L2 cache partition on the A100.
    * Since the dataset fits entirely into the L2 cache, the effective bandwidth exceeds the DRAM bandwidth limit and reflects L2 cache bandwidth instead.

## Stencil Code Optimization - Collapse

The benchmarks settle it: you need *tens of thousands* of threads to saturate the compute units, and *hundreds of thousands* to saturate the memory system (without any other tricks or more complicated workloads per thread).
Glancing back at the stencil code in [stencil-2d-omp-target-v1](../src/stencil-2d/stencil-2d-omp-target-v1.cpp), our PhD candidate realizes they only parallelize the outer loop which only has 8190 iterations, nowhere near enough parallelism.

The OpenMP course suggested the proper fix: split the construct in two, mapping team parallelism to the outer loop and thread parallelism to the inner one.
That sounds like real work.
So they ask their favorite AI instead, and are cheerfully told about the `collapse` clause, which saves a lot of typing.
It is not *strictly* the same thing, as the AI neglects to dwell on and our PhD candidate declines to ask, but who really worries about the fine print?

The updated code is available at [stencil-2d-omp-target-v2](../src/stencil-2d/stencil-2d-omp-target-v2.cpp), and can be compiled, executed and profiled with the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v2.cpp -o ../build/stencil-2d-omp-target-v2

In [ ]:
!../build/stencil-2d-omp-target-v2 double 8192 8192 2 256

In [ ]:
!ncu --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (524033, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.21
    SM Frequency                    Ghz         1.09
    Elapsed Cycles                cycle    1,067,038
    Memory Throughput                 %        70.20
    DRAM Throughput                   %        70.20
    Duration                         us       974.85
    L1/TEX Cache Throughput           %        27.55
    L2 Cache Throughput               %        80.21
    SM Active Cycles              cycle 1,063,981.31
    Compute (SM) Throughput           %        63.71
    ----------------------- ----------- ------------

    INF   Compute and Memory are well-balanced: To reduce runtime, both computation and memory traffic must be reduced. 
          Check both the Compute Workload Analysis and Memory Workload Analysis sections.                               
```

### Evaluation

That already looks a lot better.
*Now* that is a speed-up worth writing up.

Still, a nagging curiosity remains: how accurate are those *estimated* bandwidth numbers the code prints?
Nsight Compute reports percentages of peak, which is a start, but to turn a raw measurement into a percentage, it must already know the true, measured value.
Surely there is a way to ask it for that directly.
Digging through the public repository of a colleague who once did something similar, our PhD candidate finds exactly that: a way to make Nsight Compute report very specific pieces of information.

# Profiling Metrics

This approach looks very specific, our PhD candidate notes, and probably won't be useful for general performance analysis and engineering.
But for building automated pipelines that capture isolated performance metrics, this might be really helpful.
It also keeps the profiling overheads low.

The details are available for the [metrics structure](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-structure), and as a [metrics reference](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference), both part of NVIDIA's documentation.
`ncu --list-metrics` can also be used to obtain a list of available metrics - an example output is included in [profiles/metrics.txt](../profiles/metrics.txt).

The list is quite long and not particularly easy to navigate.
We give our PhD candidate a few hours to sift through the list and then look at the most relevant metrics they identified for their application:
* `sm__warps_active.avg.pct_of_peak_sustained_active` displays the achieved occupancy.
* `dram__bytes_read` and `dram__bytes_write` correspond to the bytes read from and written to DRAM.
    This can be extended with `.sum` to obtain the total volume and `.sum.per_second` to obtain the bandwidth.
* `smsp__sass_thread_inst_executed_op_{dadd, dmul, dfma}_pred_on.sum` represents the total executed additions, multiplications and fused multiply-adds in double precision.
    The total number of FLOPs in double precision can be computed with `dadd + dmul + 2 * dfma`.
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_{ld, st}.sum` can be used to query L2 load and store volumes.

Metrics to be measured can be provided to Nsight Compute on the command line using the optional argument `--metrics`:

In [ ]:
%%bash

ncu \
    --launch-skip 2 --launch-count 1 \
    --metrics \
sm__warps_active.avg.pct_of_peak_sustained_active,\
dram__bytes_read.sum,dram__bytes_write.sum,\
dram__bytes_read.sum.per_second,dram__bytes_write.sum.per_second,\
smsp__sass_thread_inst_executed_op_dadd_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dmul_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dfma_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second,\
smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second,\
smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second\
    ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (524033, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: Command line profiler metrics
    -------------------------------------------------------------- ----------- ------------
    Metric Name                                                    Metric Unit Metric Value
    -------------------------------------------------------------- ----------- ------------
    dram__bytes_read.sum                                                 Mbyte       537.39
    dram__bytes_read.sum.per_second                                    Gbyte/s       551.20
    dram__bytes_write.sum                                                Mbyte       526.85
    dram__bytes_write.sum.per_second                                   Gbyte/s       540.39
    sm__warps_active.avg.pct_of_peak_sustained_active                        %        54.94
    smsp__sass_thread_inst_executed_op_dadd_pred_on.sum                   inst            0
    smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second      inst/s            0
    smsp__sass_thread_inst_executed_op_dfma_pred_on.sum                   inst  201,228,300
    smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second     inst/ns       206.40
    smsp__sass_thread_inst_executed_op_dmul_pred_on.sum                   inst   67,076,100
    smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second     inst/ns        68.80
    -------------------------------------------------------------- ----------- ------------
```

### General Guidelines

They also note down some general guidelines for using metrics:
* For counting metrics (e.g. bytes transferred) the structure is usually `metric.sum`.
* For corresponding rates this can be extended to `metric.sum.per_second`.
* Many metrics can be recomputed in terms of percentage of theoretical peak performance with `metric.sum.pct_of_peak_sustained_elapsed`.

## Nsight Compute Sets and Sections

Scrolling a little further in that same repository, they spot another convenience: *sets* and *sections*, a shorthand for requesting whole families of metrics at once, without typing out cryptic metric names one by one.

Providing a custom set of metrics can be quite useful, but also cumbersome.
A more coarse-grained way of telling Nsight Compute what to measure is using **sections** (we've seen examples for them before) or **sets** - both effectively represent pre-defined metric lists.

Available sections can be queried with `ncu --list-sections`, available sets with `ncu --list-sets`.
Exemplary outputs are available in [profiles/sections.txt](../profiles/sections.txt) and [profiles/sets.txt](../profiles/sets.txt).
To use them, additional flags are passed to Nsight Compute.

In [ ]:
!ncu --section=SpeedOfLight --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

In [ ]:
!ncu --set=roofline --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

# Nsight Compute GUI

The command line is unbeatable for automated, scripted profiling.
But our PhD candidate is more of a visual person, and squinting at ASCII tables is starting to wear thin.
So they do the obvious thing and open the profile data in the Nsight Compute GUI.\
They follow a similar pattern as before:
* Profiling of the application with suitable sections/sets remotely.
* Downloading the profile data.
* Opening it locally.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v2.cpp -o ../build/stencil-2d-omp-target-v2

In [ ]:
!../build/stencil-2d-omp-target-v2 double 8192 8192 2 256

In [ ]:
!ncu --set full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-omp-target-v2 --force-overwrite ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

Download the obtained profile file `stencil-2d-omp-target-v2.ncu-rep` directly from the `profiles` folder, or by **shift + right-clicking** this [link](../profiles/stencil-2d-omp-target-v2.ncu-rep).
Afterwards, open it up with your local installation of Nsight Compute.

The volume of information is once again quite overwhelming when using the tool for the first time.
Start by double-clicking on (one of) the displayed kernels.
Take a moment to explore the different sections.
You can expand them by clicking the little triangle next to them.
If you are unsure about certain terms you can check short documentation snippets in the form of tooltips by hovering over what you want to know more about.

Let's focus on the **Memory Workload Analysis** section first.
It reveals that the data volume read from the L2 cache is quite high compared to the one from DRAM (about a factor of 3).

Why should the same computation move so much more data from L2 than from DRAM?
This is not a question a table will answer.
Our PhD candidate gets up, walks to the nearest whiteboard, and starts sketching how the threads actually touch memory.
On the right are the three schematic sketches they drew, showing the data read and written during stencil updates.

<img align="right" src="img/stencil-2d-single-update.png" alt="Single Stencil Update in a 2D Grid" width="300px"/>

First, a single stencil update is shown.
It is evident that one element is updated (written) and that this requires reading four neighbors.

<img align="right" src="img/stencil-2d-openmp.png" alt="Stencil Updates in a 2D Grid with a 1D Thread Decomposition" width="300px"/>

Following the parallelization employed by OpenMP results in reading about three times as many data elements as there are updates, or $3 \cdot blockSize + 2$.
This also corresponds to about three times as much data being read from L2 than written, which fits the measured values.

<img align="right" src="img/stencil-2d-cuda.png" alt="Stencil Updates in a 2D Grid with a 2D Thread Decomposition" width="300px"/>

To remedy this behavior, using (spatial) blocking techniques can be one option.
With this optimization, the volume of data read can then be approximated by $blockSize.x \cdot blockSize.y + 2 \cdot blockSize.x + 2 \cdot blockSize.y$.

# Stencil Code Optimization - Spatial Blocking

The sketch makes the diagnosis obvious: a 1D thread decomposition is a poor fit for a 2D stencil.
What is needed is *spatial blocking*.
This is achievable in OpenMP, by splitting the 2D loop into a 2D loop over blocks with 2D loops over threads inside, four nested loops in total, which, once again, sounds like a lot of work.

A colleague ambling past is drawn in by the tidy whiteboard art and asks what it is all about.
After a patient explanation, they simply shrug: *"Sounds complicated. I'd just write it in CUDA - you get a 2D thread decomposition, and the spatial blocking basically comes for free."*
Intrigued (and allergic to four nested loops), our PhD candidate decides to give CUDA a shot.

[stencil-2d-cuda-v3](../src/stencil-2d/stencil-2d-cuda-v3.cu) implements the approach detailed above.
As usual, the next cells can be used to compile and execute it.
Also note the switch from `nvc++` to `nvcc`.

In [ ]:
!nvcc -O3 -std=c++17 ../src/stencil-2d/stencil-2d-cuda-v3.cu -o ../build/stencil-2d-cuda-v3

In [ ]:
!../build/stencil-2d-cuda-v3 double 8192 8192 2 256

Oh no.
Not again.
The CUDA version is *way* too slow, and our PhD candidate is suddenly having vivid flashbacks to the opening act of this saga.
The prime suspect is the same as before, unnecessary data transfers, so, a little wiser now, they reach straight for the profiler instead of guessing.

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-cuda-v3 --force-overwrite=true ../build/stencil-2d-cuda-v3 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)       Min (ns)       Max (ns)       StdDev (ns)             Name         
 --------  ---------------  ---------  ---------------  ---------------  -----------  --------------  ---------------  ----------------------
     93.0   12,534,381,338          2  6,267,190,669.0  6,267,190,669.0   97,537,319  12,436,844,019  8,725,207,442.7  cudaDeviceSynchronize 
      4.9      659,005,370          2    329,502,685.0    329,502,685.0  249,682,571     409,322,799    112,882,687.8  cudaMallocHost        
      1.4      190,871,613          2     95,435,806.5     95,435,806.5   94,819,518      96,052,095        871,563.6  cudaFreeHost          
      0.6       82,075,719          4     20,518,929.8     20,543,053.0   20,397,762      20,591,851         92,116.8  cudaMemcpy            
      0.0        2,573,072          1      2,573,072.0      2,573,072.0    2,573,072       2,573,072              0.0  cuLibraryLoadData     
      0.0        1,582,729          2        791,364.5        791,364.5      753,208         829,521         53,961.4  cudaFree              
      0.0        1,446,727          2        723,363.5        723,363.5      415,472       1,031,255        435,424.3  cudaMalloc            
      0.0          945,464        258          3,664.6          2,745.0        2,625         198,999         12,229.3  cudaLaunchKernel      
      0.0           26,297        258            101.9             91.0           80             842             55.0  cuKernelGetName       
      0.0            3,477          1          3,477.0          3,477.0        3,477           3,477              0.0  cuModuleGetLoadingMode
      0.0              691          1            691.0            691.0          691             691              0.0  cuLibraryGetKernel    
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)                                   Name                                 
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  ----------------------------------------------------------------------
    100.0   12,534,924,746        258  48,584,979.6  48,569,369.0  47,726,661  49,504,046    380,949.7  void stencil2d<double>(const T1 *, T1 *, unsigned long, unsigned long)
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     50.2       41,093,454      2  20,546,727.0  20,546,727.0  20,524,855  20,568,599     30,931.7  [CUDA memcpy Device-to-Host]
     49.8       40,762,855      2  20,381,427.5  20,381,427.5  20,282,643  20,480,212    139,702.4  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

Curious - the transfers are *not* the problem this time.
Whatever is going wrong must be happening *inside* the kernel itself, since every single one of its launches is now incredibly slow.
That is precisely the kind of question Nsight Compute exists to answer.

**Note** we use `--set=full` here to capture as much data as possible.
Newer versions of Nsight Compute (>= 2026) capture *a lot* of data - use with care.

In [ ]:
!ncu --set full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-cuda-v3 --force-overwrite ../build/stencil-2d-cuda-v3 double 8192 8192 2 2

After downloading the [report file](../profiles/stencil-2d-cuda-v3.ncu-rep) and opening it locally, we compare this profile with the last OpenMP one.
To do so, we can click on `Compare` at the top right and then on `Add Baseline`.
After opening the profile for the CUDA version, we once again focus on the memory workload section.
We can see multiple effects:
* The data volume read from L2 is now a lot lower (about 2x)
    - Cool, the optimization works.
* The data volume read from and written to DRAM is now about zero
    - This is ... *weird*.
* There is a new data path to system memory
    - Huh, that is definitely not good.

# Final Stencil Code Optimization

The previous analysis reports data transfers from and to system memory, or in other words the host memory, instead of using the GPU memory.
Reviewing the code reveals the culprit: instead of device pointers, host pointers are passed to the kernel.
In a sense, their intuition about shifting the bottleneck to the PCIe bus was correct - but not as bulk transfers which would show up in Nsight Systems, but rather as a massive number of accesses with tiny granularity.

Our PhD candidate is quietly relieved on two counts: first, that nobody witnessed this particular rookie mistake, and second, that the fix turns out to be delightfully simple.
The updated code version is available at [stencil-2d-cuda-v4](../src/stencil-2d/stencil-2d-cuda-v4.cu).

In [ ]:
!nvcc -O3 -std=c++17 ../src/stencil-2d/stencil-2d-cuda-v4.cu -o ../build/stencil-2d-cuda-v4

In [ ]:
!../build/stencil-2d-cuda-v4 double 8192 8192 2 256

The performance of the application is now quite close to the measured limit.
Further optimization of the *implementation* will likely not bring any performance benefits.
To further accelerate, *algorithmic optimizations* are required, e.g. more advanced (temporal) blocking or switching to a different numerical solver algorithm altogether.

Nsight Compute also shows traces of this when checking the roofline section (download the profile [here](../profiles/stencil-2d-cuda-v4.ncu-rep)) after running the next cell.
You can review it in the **GPU Speed of Light Throughput** section by choosing **Roofline Double Precision** in the top right drop-down.

In [ ]:
!ncu --set=full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-cuda-v4 --force-overwrite ../build/stencil-2d-cuda-v4 double 8192 8192 2 2

Roofline models are an extended version of the discussed bottleneck model.
Further details are beyond the scope of this introduction but, generally speaking: points in the chart that are close to the roofline indicate that the profiled code utilizes the hardware well.

To complete the performance evaluation, also check the Nsight Systems timeline (download the profile [here](../profiles/stencil-2d-cuda-v4.nsys-rep)) after running the below cell.

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-cuda-v4 --force-overwrite=true ../build/stencil-2d-cuda-v4 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)       Med (ns)      Min (ns)     Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  -------------  -------------  -----------  -----------  -------------  ----------------------
     58.0      668,535,207          2  334,267,603.5  334,267,603.5  247,073,353  421,461,854  123,311,291.6  cudaMallocHost        
     18.4      212,479,643          2  106,239,821.5  106,239,821.5    1,648,063  210,831,580  147,915,083.4  cudaDeviceSynchronize 
     16.0      184,315,843          2   92,157,921.5   92,157,921.5   90,414,387   93,901,456    2,465,730.1  cudaFreeHost          
      7.1       82,004,935          4   20,501,233.8   20,481,596.5   20,308,870   20,732,872      175,362.8  cudaMemcpy            
      0.2        2,215,643          1    2,215,643.0    2,215,643.0    2,215,643    2,215,643            0.0  cuLibraryLoadData     
      0.1        1,712,773          2      856,386.5      856,386.5      847,824      864,949       12,109.2  cudaFree              
      0.1        1,330,294          2      665,147.0      665,147.0      420,916      909,378      345,394.8  cudaMalloc            
      0.1          926,358        258        3,590.5        2,715.0        2,635      189,181       11,630.5  cudaLaunchKernel      
      0.0           25,655        258           99.4          100.0           80          371           22.2  cuKernelGetName       
      0.0            3,197          1        3,197.0        3,197.0        3,197        3,197            0.0  cuModuleGetLoadingMode
      0.0              561          1          561.0          561.0          561          561            0.0  cuLibraryGetKernel    
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances  Avg (ns)   Med (ns)   Min (ns)  Max (ns)  StdDev (ns)                                   Name                                 
 --------  ---------------  ---------  ---------  ---------  --------  --------  -----------  ----------------------------------------------------------------------
    100.0      212,844,996        258  824,980.6  825,098.0   815,754   832,170      3,494.0  void stencil2d<double>(const T1 *, T1 *, unsigned long, unsigned long)
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     50.3       41,190,884      2  20,595,442.0  20,595,442.0  20,485,265  20,705,619    155,813.8  [CUDA memcpy Device-to-Host]
     49.7       40,643,838      2  20,321,919.0  20,321,919.0  20,292,239  20,351,599     41,973.9  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

# Next Steps

And so, at last, our PhD candidate writes the paper, only to discover, at the worst possible moment, that stencil codes have been studied intensely for several decades and that a fast stencil kernel is, regrettably, not a novel contribution.
Squeezing out more performance now would require *algorithmic* changes, such as temporal blocking, diamond tiling, or a fundamentally more efficient solver algorithm, rather than more profiling.
Sensing that this is a problem for future-them, they close the laptop and go for a well-earned walk.

On that walk, our PhD candidate resolves to properly learn GPU performance analysis before wrestling with algorithmic optimizations.
The sections below collect what they picked up along the way: a short catalogue of common performance patterns to watch out for, and pointers to material they used to gain knowledge.

**Common Patterns**

In cases where performance is not close to theoretical limits there are often common patterns to be observed that provide an explanation:
* **Thread divergence**: threads of the same warp take different code paths.
* **(Un-)coalesced memory accesses**: consecutive threads access (non-)consecutive memory locations.
* **Kernel launch overhead**: many short-running kernels lead to a low ratio of time spent in kernels over total execution time.
* **Low occupancy**: lack of parallelism, and/or overusing available resources lead to a low occupancy.

**Additional Material**

* Many computing centers (regional, national and beyond) around the world offer excellent trainings, often online and free of charge for members of academia.
* The official NVIDIA documentation, in particular the [CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html) and the [CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/), covers both the underlying concepts and the practical details.
* NVIDIA's [Nsight tools playlist](https://www.youtube.com/playlist?list=PL5B692fm6--ukF8S7ul5NmceZhXLRv_lR) walks through the profiling tools used in this tutorial in greater depth.
* [GTC](https://www.nvidia.com/gtc/) offers a wide spectrum of material around GPU computing. There is also an [on-demand archive](https://www.nvidia.com/on-demand/) with thousands of videos.